## SRP154817

**paper:** [PMID: 30192940](https://pmc.ncbi.nlm.nih.gov/articles/PMC6204425/) - Comparative genomics and transcriptomics of Chrysolophus provide insights into the evolution of complex plumage coloration, 2018

**date, curator:** 2026-03-26, Sara Carsanaro

**notes**
* per extended methods the animals are 2 year old

### annotation summary

In [31]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Tail feather follicles,UBERON:0011782,feather follicle,missing child term
1,Mantle feather follicles,UBERON:0011782,feather follicle,missing child term
2,Belly feather follicles,UBERON:0011782,feather follicle,missing child term
3,Nape feather follicles,UBERON:0011782,feather follicle,missing child term
4,Rump feather follicles,UBERON:0011782,feather follicle,missing child term


In [32]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,adult,UBERON:0000113,post-juvenile adult stage,missing child term


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP154817"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|█████████████████████████████████████████████| 6/6 [00:06<00:00,  1.10s/it]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX4416559,SRP154817,Illumina HiSeq 2000,SRS3569702,,,,,,Tail feather follicles,adult,,,,M,,,9089,,,,,,P-F-Tail,SAMN08398522,,,,,,,,,,,26/03/2026,"The total RNA samples are first treated with DNase I to degrade any possible DNA contamination. Then the mRNA is enriched by using the oligo(dT) magnetic beads. Mixed with the fragmentation buffer, the mRNA is fragmented into short fragments (about 200 bp). Then the first strand of cDNA is synthesized by using random hexamer-primer. Buffer, dNTPs, RNase H and DNA polymerase I are added to synthesize the second strand. The double strand cDNA is purified with magnetic beads. End reparation and 3'-end single nucleotide A (adenine) addition is then performed. Finally, sequencing adaptors are ligated to the fragments. The fragments are enriched by PCR amplification. During the QC step, Agilent 2100 Bioanaylzer and ABI StepOnePlus Real-Time PCR System are used to qualify and quantify of the sample library. The library products are ready for sequencing via Illumina HiSeqTM 2000 or other sequencer when necessary.",,P-F-Tail,,,,adult,,TRANSCRIPTOMIC,RANDOM
1,SRX4416558,SRP154817,Illumina HiSeq 2000,SRS3569701,,,,,,Mantle feather follicles,adult,,,,M,,,9089,,,,,,P-F-Mantle,SAMN08398521,,,,,,,,,,,26/03/2026,"The total RNA samples are first treated with DNase I to degrade any possible DNA contamination. Then the mRNA is enriched by using the oligo(dT) magnetic beads. Mixed with the fragmentation buffer, the mRNA is fragmented into short fragments (about 200 bp). Then the first strand of cDNA is synthesized by using random hexamer-primer. Buffer, dNTPs, RNase H and DNA polymerase I are added to synthesize the second strand. The double strand cDNA is purified with magnetic beads. End reparation and 3'-end single nucleotide A (adenine) addition is then performed. Finally, sequencing adaptors are ligated to the fragments. The fragments are enriched by PCR amplification. During the QC step, Agilent 2100 Bioanaylzer and ABI StepOnePlus Real-Time PCR System are used to qualify and quantify of the sample library. The library products are ready for sequencing via Illumina HiSeqTM 2000 or other sequencer when necessary.",,P-F-Mantle,,,,adult,,TRANSCRIPTOMIC,RANDOM
2,SRX4416557,SRP154817,Illumina HiSeq 2000,SRS3569700,,,,,,Belly feather follicles,adult,,,,M,,,9089,,,,,,P-F-Belly,SAMN08398520,,,,,,,,,,,26/03/2026,"The total RNA samples are first treated with DNase I to degrade any possible DNA contamination. Then the mRNA is enriched by using the oligo(dT) magnetic beads. Mixed with the fragmentation buffer, the mRNA is fragmented into short fragments (about 200 bp). Then the first strand of cDNA is synthesized by using random hexamer-primer. Buffer, dNTPs, RNase H and DNA polymerase I are added to synthesize the second strand. The double strand cDNA is purified with magnetic beads. End reparation and 3'-end single nucleotide A (adenine) addition is then performed. Finally, sequencing adaptors are ligated to the fragments. The fragments are enriched by PCR amplification. During the QC step, Agilent 2100 Bioanaylzer and ABI StepOnePlus Real-Time PCR System are used to qualify and quantify of the sample library. The library products are ready for sequencing via Illumina HiSeqTM 2000 or other sequencer when necessary.",,P-F-Belly,,,,adult,,TRANSCRIPTOMIC,RANDOM
3,SRX4416556,SRP154817,Illumina HiSeq 2000,SRS3569699,,,,,,Nape feather follicles,adult,,,,M,,,9089,,,,,,P-

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['Belly feather follicles' 'Mantle feather follicles'
 'Nape feather follicles' 'Rump feather follicles'
 'Tail feather follicles']


In [6]:

# all
library.loc[:,'anatId'] = 'UBERON:0011782'
library.loc[:,'anatName'] = 'feather follicle'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'missing child term'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX4416559,SRP154817,Illumina HiSeq 2000,SRS3569702,UBERON:0011782,feather follicle,,,,Tail feather follicles,adult,missing child term,not documented,,M,,,9089,,,,,,P-F-Tail,SAMN08398522,,,,,,,,,,,26/03/2026,"The total RNA samples are first treated with DNase I to degrade any possible DNA contamination. Then the mRNA is enriched by using the oligo(dT) magnetic beads. Mixed with the fragmentation buffer, the mRNA is fragmented into short fragments (about 200 bp). Then the first strand of cDNA is synthesized by using random hexamer-primer. Buffer, dNTPs, RNase H and DNA polymerase I are added to synthesize the second strand. The double strand cDNA is purified with magnetic beads. End reparation and 3'-end single nucleotide A (adenine) addition is then performed. Finally, sequencing adaptors are ligated to the fragments. The fragments are enriched by PCR amplification. During the QC step, Agilent 2100 Bioanaylzer and ABI StepOnePlus Real-Time PCR System are used to qualify and quantify of the sample library. The library products are ready for sequencing via Illumina HiSeqTM 2000 or other sequencer when necessary.",,P-F-Tail,,,,adult,,TRANSCRIPTOMIC,RANDOM
1,SRX4416558,SRP154817,Illumina HiSeq 2000,SRS3569701,UBERON:0011782,feather follicle,,,,Mantle feather follicles,adult,missing child term,not documented,,M,,,9089,,,,,,P-F-Mantle,SAMN08398521,,,,,,,,,,,26/03/2026,"The total RNA samples are first treated with DNase I to degrade any possible DNA contamination. Then the mRNA is enriched by using the oligo(dT) magnetic beads. Mixed with the fragmentation buffer, the mRNA is fragmented into short fragments (about 200 bp). Then the first strand of cDNA is synthesized by using random hexamer-primer. Buffer, dNTPs, RNase H and DNA polymerase I are added to synthesize the second strand. The double strand cDNA is purified with magnetic beads. End reparation and 3'-end single nucleotide A (adenine) addition is then performed. Finally, sequencing adaptors are ligated to the fragments. The fragments are enriched by PCR amplification. During the QC step, Agilent 2100 Bioanaylzer and ABI StepOnePlus Real-Time PCR System are used to qualify and quantify of the sample library. The library products are ready for sequencing via Illumina HiSeqTM 2000 or other sequencer when necessary.",,P-F-Mantle,,,,adult,,TRANSCRIPTOMIC,RANDOM
2,SRX4416557,SRP154817,Illumina HiSeq 2000,SRS3569700,UBERON:0011782,feather follicle,,,,Belly feather follicles,adult,missing child term,not documented,,M,,,9089,,,,,,P-F-Belly,SAMN08398520,,,,,,,,,,,26/03/2026,"The total RNA samples are first treated with DNase I to degrade any possible DNA contamination. Then the mRNA is enriched by using the oligo(dT) magnetic beads. Mixed with the fragmentation buffer, the mRNA is fragmented into short fragments (about 200 bp). Then the first strand of cDNA is synthesized by using random hexamer-primer. Buffer, dNTPs, RNase H and DNA polymerase I are added to synthesize the second strand. The double strand cDNA is purified with magnetic beads. End reparation and 3'-end single nucleotide A (adenine) addition is then performed. Finally, sequencing adaptors are ligated to the fragments. The fragments are enriched by PCR amplification. During the QC step, Agilent 2100 Bioanaylzer and ABI StepOnePlus Real-Time PCR System are used to qualify and quantify of the sample library. The library products are ready for sequencing via Illumina HiSeqTM 2000 

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['adult']


In [8]:
# all
library.loc[:,'stageId'] = 'UBERON:0000113'
library.loc[:,'stageName'] = 'post-juvenile adult stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'missing child term'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX4416559,SRP154817,Illumina HiSeq 2000,SRS3569702,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Tail feather follicles,adult,missing child term,not documented,missing child term,M,,,9089,,,,,,P-F-Tail,SAMN08398522,,,,,,,,,,,26/03/2026,"The total RNA samples are first treated with DNase I to degrade any possible DNA contamination. Then the mRNA is enriched by using the oligo(dT) magnetic beads. Mixed with the fragmentation buffer, the mRNA is fragmented into short fragments (about 200 bp). Then the first strand of cDNA is synthesized by using random hexamer-primer. Buffer, dNTPs, RNase H and DNA polymerase I are added to synthesize the second strand. The double strand cDNA is purified with magnetic beads. End reparation and 3'-end single nucleotide A (adenine) addition is then performed. Finally, sequencing adaptors are ligated to the fragments. The fragments are enriched by PCR amplification. During the QC step, Agilent 2100 Bioanaylzer and ABI StepOnePlus Real-Time PCR System are used to qualify and quantify of the sample library. The library products are ready for sequencing via Illumina HiSeqTM 2000 or other sequencer when necessary.",,P-F-Tail,,,,adult,,TRANSCRIPTOMIC,RANDOM
1,SRX4416558,SRP154817,Illumina HiSeq 2000,SRS3569701,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Mantle feather follicles,adult,missing child term,not documented,missing child term,M,,,9089,,,,,,P-F-Mantle,SAMN08398521,,,,,,,,,,,26/03/2026,"The total RNA samples are first treated with DNase I to degrade any possible DNA contamination. Then the mRNA is enriched by using the oligo(dT) magnetic beads. Mixed with the fragmentation buffer, the mRNA is fragmented into short fragments (about 200 bp). Then the first strand of cDNA is synthesized by using random hexamer-primer. Buffer, dNTPs, RNase H and DNA polymerase I are added to synthesize the second strand. The double strand cDNA is purified with magnetic beads. End reparation and 3'-end single nucleotide A (adenine) addition is then performed. Finally, sequencing adaptors are ligated to the fragments. The fragments are enriched by PCR amplification. During the QC step, Agilent 2100 Bioanaylzer and ABI StepOnePlus Real-Time PCR System are used to qualify and quantify of the sample library. The library products are ready for sequencing via Illumina HiSeqTM 2000 or other sequencer when necessary.",,P-F-Mantle,,,,adult,,TRANSCRIPTOMIC,RANDOM
2,SRX4416557,SRP154817,Illumina HiSeq 2000,SRS3569700,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Belly feather follicles,adult,missing child term,not documented,missing child term,M,,,9089,,,,,,P-F-Belly,SAMN08398520,,,,,,,,,,,26/03/2026,"The total RNA samples are first treated with DNase I to degrade any possible DNA contamination. Then the mRNA is enriched by using the oligo(dT) magnetic beads. Mixed with the fragmentation buffer, the mRNA is fragmented into short fragments (about 200 bp). Then the first strand of cDNA is synthesized by using random hexamer-primer. Buffer, dNTPs, RNase H and DNA polymerase I are added to synthesize the second strand. The double strand cDNA is purified with magnetic beads. End reparation and 3'-end single nucleotide A (adenine) addition is then performed. Finally, sequencing adaptors are ligated to the fragments. The fragments are enriched by PCR amplification. During the QC step, Agilent 2100 Bioanaylze

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [9]:
# making these variables because we use them again in the experiment file
my_protocol = 'TruSeq RNA Sample Preparation Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX4416559,SRP154817,Illumina HiSeq 2000,SRS3569702,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Tail feather follicles,adult,missing child term,not documented,missing child term,M,,,9089,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,P-F-Tail,SAMN08398522,,,,,,,,,,,26/03/2026,"The total RNA samples are first treated with DNase I to degrade any possible DNA contamination. Then the mRNA is enriched by using the oligo(dT) magnetic beads. Mixed with the fragmentation buffer, the mRNA is fragmented into short fragments (about 200 bp). Then the first strand of cDNA is synthesized by using random hexamer-primer. Buffer, dNTPs, RNase H and DNA polymerase I are added to synthesize the second strand. The double strand cDNA is purified with magnetic beads. End reparation and 3'-end single nucleotide A (adenine) addition is then performed. Finally, sequencing adaptors are ligated to the fragments. The fragments are enriched by PCR amplification. During the QC step, Agilent 2100 Bioanaylzer and ABI StepOnePlus Real-Time PCR System are used to qualify and quantify of the sample library. The library products are ready for sequencing via Illumina HiSeqTM 2000 or other sequencer when necessary.",,P-F-Tail,,,,adult,,TRANSCRIPTOMIC,RANDOM
1,SRX4416558,SRP154817,Illumina HiSeq 2000,SRS3569701,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Mantle feather follicles,adult,missing child term,not documented,missing child term,M,,,9089,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,P-F-Mantle,SAMN08398521,,,,,,,,,,,26/03/2026,"The total RNA samples are first treated with DNase I to degrade any possible DNA contamination. Then the mRNA is enriched by using the oligo(dT) magnetic beads. Mixed with the fragmentation buffer, the mRNA is fragmented into short fragments (about 200 bp). Then the first strand of cDNA is synthesized by using random hexamer-primer. Buffer, dNTPs, RNase H and DNA polymerase I are added to synthesize the second strand. The double strand cDNA is purified with magnetic beads. End reparation and 3'-end single nucleotide A (adenine) addition is then performed. Finally, sequencing adaptors are ligated to the fragments. The fragments are enriched by PCR amplification. During the QC step, Agilent 2100 Bioanaylzer and ABI StepOnePlus Real-Time PCR System are used to qualify and quantify of the sample library. The library products are ready for sequencing via Illumina HiSeqTM 2000 or other sequencer when necessary.",,P-F-Mantle,,,,adult,,TRANSCRIPTOMIC,RANDOM
2,SRX4416557,SRP154817,Illumina HiSeq 2000,SRS3569700,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Belly feather follicles,adult,missing child term,not documented,missing child term,M,,,9089,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,P-F-Belly,SAMN08398520,,,,,,,,,,,26/03/2026,"The total RNA samples are first treated with DNase I to degrade any possible DNA contamination. Then the mRNA is enriched by using the oligo(dT) magnetic beads. Mixed with the fragmentation buffer, the mRNA is fragmented into short fragments (about 200 bp). Then the first strand of cDNA is synthesized by using random hexamer-primer. Buffer, dNTPs, RNase H and DNA polymerase I are added to synthesize the second strand. The double strand cDNA is purified with magnetic beads. End reparation and 3'-end single nucleotide A (adenine) addition is then performed. Fina

#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [11]:
library.loc[:,'sampleAge_value'] = '2'
library.loc[:,'sampleAge_unit'] = 'year'

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX4416559,SRP154817,Illumina HiSeq 2000,SRS3569702,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Tail feather follicles,adult,missing child term,not documented,missing child term,M,,,9089,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,P-F-Tail,SAMN08398522,2,year,,,,,,,,,26/03/2026,"The total RNA samples are first treated with DNase I to degrade any possible DNA contamination. Then the mRNA is enriched by using the oligo(dT) magnetic beads. Mixed with the fragmentation buffer, the mRNA is fragmented into short fragments (about 200 bp). Then the first strand of cDNA is synthesized by using random hexamer-primer. Buffer, dNTPs, RNase H and DNA polymerase I are added to synthesize the second strand. The double strand cDNA is purified with magnetic beads. End reparation and 3'-end single nucleotide A (adenine) addition is then performed. Finally, sequencing adaptors are ligated to the fragments. The fragments are enriched by PCR amplification. During the QC step, Agilent 2100 Bioanaylzer and ABI StepOnePlus Real-Time PCR System are used to qualify and quantify of the sample library. The library products are ready for sequencing via Illumina HiSeqTM 2000 or other sequencer when necessary.",,P-F-Tail,,,,adult,,TRANSCRIPTOMIC,RANDOM
1,SRX4416558,SRP154817,Illumina HiSeq 2000,SRS3569701,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Mantle feather follicles,adult,missing child term,not documented,missing child term,M,,,9089,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,P-F-Mantle,SAMN08398521,2,year,,,,,,,,,26/03/2026,"The total RNA samples are first treated with DNase I to degrade any possible DNA contamination. Then the mRNA is enriched by using the oligo(dT) magnetic beads. Mixed with the fragmentation buffer, the mRNA is fragmented into short fragments (about 200 bp). Then the first strand of cDNA is synthesized by using random hexamer-primer. Buffer, dNTPs, RNase H and DNA polymerase I are added to synthesize the second strand. The double strand cDNA is purified with magnetic beads. End reparation and 3'-end single nucleotide A (adenine) addition is then performed. Finally, sequencing adaptors are ligated to the fragments. The fragments are enriched by PCR amplification. During the QC step, Agilent 2100 Bioanaylzer and ABI StepOnePlus Real-Time PCR System are used to qualify and quantify of the sample library. The library products are ready for sequencing via Illumina HiSeqTM 2000 or other sequencer when necessary.",,P-F-Mantle,,,,adult,,TRANSCRIPTOMIC,RANDOM
2,SRX4416557,SRP154817,Illumina HiSeq 2000,SRS3569700,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Belly feather follicles,adult,missing child term,not documented,missing child term,M,,,9089,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,P-F-Belly,SAMN08398520,2,year,,,,,,,,,26/03/2026,"The total RNA samples are first treated with DNase I to degrade any possible DNA contamination. Then the mRNA is enriched by using the oligo(dT) magnetic beads. Mixed with the fragmentation buffer, the mRNA is fragmented into short fragments (about 200 bp). Then the first strand of cDNA is synthesized by using random hexamer-primer. Buffer, dNTPs, RNase H and DNA polymerase I are added to synthesize the second strand. The double strand cDNA is purified with magnetic beads. End reparation and 3'-end single nucleotide A (adenine) addition is then 

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [12]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-03-26'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX4416559,SRP154817,Illumina HiSeq 2000,SRS3569702,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Tail feather follicles,adult,missing child term,not documented,missing child term,M,,,9089,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,P-F-Tail,SAMN08398522,2,year,,,,,,,,SAC,2026-03-26,"The total RNA samples are first treated with DNase I to degrade any possible DNA contamination. Then the mRNA is enriched by using the oligo(dT) magnetic beads. Mixed with the fragmentation buffer, the mRNA is fragmented into short fragments (about 200 bp). Then the first strand of cDNA is synthesized by using random hexamer-primer. Buffer, dNTPs, RNase H and DNA polymerase I are added to synthesize the second strand. The double strand cDNA is purified with magnetic beads. End reparation and 3'-end single nucleotide A (adenine) addition is then performed. Finally, sequencing adaptors are ligated to the fragments. The fragments are enriched by PCR amplification. During the QC step, Agilent 2100 Bioanaylzer and ABI StepOnePlus Real-Time PCR System are used to qualify and quantify of the sample library. The library products are ready for sequencing via Illumina HiSeqTM 2000 or other sequencer when necessary.",,P-F-Tail,,,,adult,,TRANSCRIPTOMIC,RANDOM
1,SRX4416558,SRP154817,Illumina HiSeq 2000,SRS3569701,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Mantle feather follicles,adult,missing child term,not documented,missing child term,M,,,9089,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,P-F-Mantle,SAMN08398521,2,year,,,,,,,,SAC,2026-03-26,"The total RNA samples are first treated with DNase I to degrade any possible DNA contamination. Then the mRNA is enriched by using the oligo(dT) magnetic beads. Mixed with the fragmentation buffer, the mRNA is fragmented into short fragments (about 200 bp). Then the first strand of cDNA is synthesized by using random hexamer-primer. Buffer, dNTPs, RNase H and DNA polymerase I are added to synthesize the second strand. The double strand cDNA is purified with magnetic beads. End reparation and 3'-end single nucleotide A (adenine) addition is then performed. Finally, sequencing adaptors are ligated to the fragments. The fragments are enriched by PCR amplification. During the QC step, Agilent 2100 Bioanaylzer and ABI StepOnePlus Real-Time PCR System are used to qualify and quantify of the sample library. The library products are ready for sequencing via Illumina HiSeqTM 2000 or other sequencer when necessary.",,P-F-Mantle,,,,adult,,TRANSCRIPTOMIC,RANDOM
2,SRX4416557,SRP154817,Illumina HiSeq 2000,SRS3569700,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Belly feather follicles,adult,missing child term,not documented,missing child term,M,,,9089,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,P-F-Belly,SAMN08398520,2,year,,,,,,,,SAC,2026-03-26,"The total RNA samples are first treated with DNase I to degrade any possible DNA contamination. Then the mRNA is enriched by using the oligo(dT) magnetic beads. Mixed with the fragmentation buffer, the mRNA is fragmented into short fragments (about 200 bp). Then the first strand of cDNA is synthesized by using random hexamer-primer. Buffer, dNTPs, RNase H and DNA polymerase I are added to synthesize the second strand. The double strand cDNA is purified with magnetic beads. End reparation and 3'-end single nucleotide A (adenine) addition

#### comments

In [13]:
library.loc[:,'comment'] = 'PMID: 30192940'

#### save complete file with correct columns

In [14]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX4416559,SRP154817,Illumina HiSeq 2000,SRS3569702,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Tail feather follicles,adult,missing child term,not documented,missing child term,M,,,9089,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,P-F-Tail,SAMN08398522,2,year,,,,,PMID: 30192940,,,SAC,2026-03-26
1,SRX4416558,SRP154817,Illumina HiSeq 2000,SRS3569701,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Mantle feather follicles,adult,missing child term,not documented,missing child term,M,,,9089,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,P-F-Mantle,SAMN08398521,2,year,,,,,PMID: 30192940,,,SAC,2026-03-26
2,SRX4416557,SRP154817,Illumina HiSeq 2000,SRS3569700,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Belly feather follicles,adult,missing child term,not documented,missing child term,M,,,9089,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,P-F-Belly,SAMN08398520,2,year,,,,,PMID: 30192940,,,SAC,2026-03-26
3,SRX4416556,SRP154817,Illumina HiSeq 2000,SRS3569699,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Nape feather follicles,adult,missing child term,not documented,missing child term,M,,,9089,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,P-F-Nape,SAMN08398519,2,year,,,,,PMID: 30192940,,,SAC,2026-03-26
4,SRX4416555,SRP154817,Illumina HiSeq 2000,SRS3569698,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Rump feather follicles,adult,missing child term,not documented,missing child term,M,,,9089,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,P-F-Rump,SAMN08398518,2,year,,,,,PMID: 30192940,,,SAC,2026-03-26
5,SRX4416554,SRP154817,Illumina HiSeq 2000,SRS3569697,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Nape feather follicles,adult,missing child term,not documented,missing child term,F,,,9089,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,P-F-Female Nape,SAMN08398505,2,year,,,,,PMID: 30192940,,,SAC,2026-03-26


### experiment annotations

In [15]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP154817,Feather transcriptome sequencing of Chrysolophus pictus,Investigate plumage coloration mechanisms in birds,SRA,,,,,,,PRJNA431701,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [16]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

6

In [17]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP154817,Feather transcriptome sequencing of Chrysolophus pictus,Investigate plumage coloration mechanisms in birds,SRA,total,Bgee 1K,6,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA431701,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [18]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '30192940'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC6204425/'
experiment.loc[:,'DOI'] = '10.1093/gigascience/giy113'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP154817,Feather transcriptome sequencing of Chrysolophus pictus,Investigate plumage coloration mechanisms in birds,SRA,total,Bgee 1K,6,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA431701,30192940,https://pmc.ncbi.nlm.nih.gov/articles/PMC6204425/,10.1093/gigascience/giy113,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [19]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [20]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [21]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [22]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [23]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
56863,SRX1797604,SRP075618,Illumina HiSeq 2000,SRS1465361,UBERON:0001417,skin of neck,UBERON:0000113,post-juvenile adult stage,,skin from nape,2,missing child term,not documented,missing child term,M,,,9089,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,S_Nape_P,SAMN05171427,2,year,,,,,PMID: 27265652,,,SAC,2026-03-26
56864,SRX1797603,SRP075618,Illumina HiSeq 2000,SRS1464835,UBERON:0001085,skin of trunk,UBERON:0000113,post-juvenile adult stage,,skin from belly,2,other,not documented,missing child term,M,,,9089,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,S_Belly_P,SAMN05171426,2,year,,,,,PMID: 27265652. they call this belly and breas...,,,SAC,2026-03-26
56865,SRX4416559,SRP154817,Illumina HiSeq 2000,SRS3569702,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Tail feather follicles,adult,missing child term,not documented,missing child term,M,,,9089,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,P-F-Tail,SAMN08398522,2,year,,,,,PMID: 30192940,,,SAC,2026-03-26
56866,SRX4416558,SRP154817,Illumina HiSeq 2000,SRS3569701,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Mantle feather follicles,adult,missing child term,not documented,missing child term,M,,,9089,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,P-F-Mantle,SAMN08398521,2,year,,,,,PMID: 30192940,,,SAC,2026-03-26
56867,SRX4416557,SRP154817,Illumina HiSeq 2000,SRS3569700,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Belly feather follicles,adult,missing child term,not documented,missing child term,M,,,9089,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,P-F-Belly,SAMN08398520,2,year,,,,,PMID: 30192940,,,SAC,2026-03-26
56868,SRX4416556,SRP154817,Illumina HiSeq 2000,SRS3569699,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Nape feather follicles,adult,missing child term,not documented,missing child term,M,,,9089,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,P-F-Nape,SAMN08398519,2,year,,,,,PMID: 30192940,,,SAC,2026-03-26
56869,SRX4416555,SRP154817,Illumina HiSeq 2000,SRS3569698,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,Rump feather follicles,adult,missing child term,not documented,missing child term,M,,,9089,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,P-F-Rump,SAMN08398518,2,year,,,,,PMID: 30192940,,,SAC,2026-03-26


In [24]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1122,SRP222887,Phasianus colchicus Genome sequencing and asse...,"The common pheasant, Phasianus colchicus, is a...",SRA,partial,Bgee 1K,5,,,,PRJNA380312,33188724,https://onlinelibrary.wiley.com/doi/10.1111/17...,10.1111/1755-0998.13296,,removed DNA libraries
1123,SRP075618,Chrysolophus pictus Transcriptome or Gene expr...,The goal of this project is to compare gene ex...,SRA,total,Bgee 1K,4,NEBNext Ultra RNA Library Prep Kit,full_length,,PRJNA322582,27265652,https://pmc.ncbi.nlm.nih.gov/articles/PMC4914577/,10.13918/j.issn.2095-8137.2016.3.144,,
1124,SRP154817,Feather transcriptome sequencing of Chrysoloph...,Investigate plumage coloration mechanisms in b...,SRA,total,Bgee 1K,6,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA431701,30192940,https://pmc.ncbi.nlm.nih.gov/articles/PMC6204425/,10.1093/gigascience/giy113,,


### add annotations to git

In [25]:
! git pull

Already up to date.


In [26]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [27]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../NCBI_output/
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [28]:
! git add $git_experiment_path $git_library_path

In [29]:
! git commit -m $commit_message_exp

[develop 86a5edc] adding annotated bulk experiment SRP154817
 2 files changed, 7 insertions(+)


In [30]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 1.86 KiB | 1.86 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: ========================================================================
remote: 
remote:      Hello all, emergency maintenance this afternoon from 13:00 to
remote:                                  14:30.
remote: 
remote: ========================================================================
remote: 
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   89b1068..86a5edc  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push